# CZU-MHAD: Per-Fold Feature Selection Comparison (Standard Methods)

This notebook compares **baseline and 3 standard feature selection methods** across all 5 LOSO folds,
sweeping **5 target feature percentages** (10%, 25%, 50%, 75%, 90%) and reporting the average
test accuracy for each target across folds.

**Methods:**
1. Baseline (all features)
2. Mutual Information (filter)
3. RFE (wrapper)
4. LASSO/L1 (embedded)


In [1]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce GTX 1080 Ti


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA, SparsePCA
try:
    import umap
except ImportError:
    umap = None  # install with: pip install umap-learn
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
PyTorch version: 2.6.0+cu124
GPU: NVIDIA GeForce GTX 1080 Ti


## Configuration

In [3]:
# Paths
FEATURE_DIR = Path("features")

# Master output directory
RESULTS_ROOT = Path("results_standard_fs")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters
N_FOLDS = 5
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

DEPTH_PCA_DIM = 512

DEPTH_DR_METHOD = 'kernel_pca'

# Feature selection hyperparameters - metaheuristic
# For standard methods (target: select ~50% of features)
# Target percentages to sweep for standard methods
TARGET_FEATURE_PERCENTAGES = [0.10, 0.25, 0.50, 0.75, 0.90]
ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso']

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")


Configuration:
  N_FOLDS: 5
  N_EPOCHS: 300
  Total methods (incl. baseline): 4


## Load Data

In [4]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

# Get feature dimensions from first sample
first_sample = X_feat[0]
MODALITY_KEYS = ['depth_feat', 'sensor_feat', 'skeleton_feat']
MODALITY_NAMES = ['depth', 'sensor', 'skeleton']

RAW_FEATURE_DIMS = {}
for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    RAW_FEATURE_DIMS[name] = first_sample[key].shape[0]

# Store per-modality arrays (N_samples x D_modality)
X_per_modality = {}
for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    X_per_modality[name] = np.array([s[key] for s in X_feat])

print(f"\nRaw feature dimensions (before PCA):")
for name, dim in RAW_FEATURE_DIMS.items():
    print(f"  {name}: {dim}")
print(f"  TOTAL: {sum(RAW_FEATURE_DIMS.values())}")

# FEATURE_DIMS will be set dynamically per-fold after PCA
# (depth dim changes if PCA is applied)
print(f"\nDepth PCA target: {DEPTH_PCA_DIM}" if DEPTH_PCA_DIM else "\nDepth PCA: disabled")


Loaded 1165 samples
Number of classes: 22
Number of subjects: 5

Raw feature dimensions (before PCA):
  depth: 5508
  sensor: 652
  skeleton: 1879
  TOTAL: 8039

Depth PCA target: 512


## Neural Network Model (Same as Original)

In [5]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [6]:
def prepare_fold_data_per_modality(X_per_modality, train_idx, val_idx, test_idx,
                                    depth_pca_dim=None):
    """
    Per-modality pipeline: split -> normalize per modality -> PCA on depth -> concatenate.
    
    Returns:
        X_train, X_val, X_test: normalized (and PCA-reduced) concatenated arrays
        feature_dims: dict of {modality: dim} AFTER PCA (needed for retention calc)
        pca_obj: fitted PCA object (or None) for reference
    """
    modality_train = {}
    modality_val = {}
    modality_test = {}
    scalers = {}
    pca_obj = None
    feature_dims = {}
    
    for name in MODALITY_NAMES:
        X_mod = X_per_modality[name]
        
        # Split
        X_tr = X_mod[train_idx]
        X_v  = X_mod[val_idx]
        X_te = X_mod[test_idx]
        
        # Per-modality normalization (fit on train only)
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_v  = scaler.transform(X_v)
        X_te = scaler.transform(X_te)
        scalers[name] = scaler
        
        # Dimensionality reduction on depth only (method chosen by DEPTH_DR_METHOD)
        if name == 'depth' and depth_pca_dim is not None and depth_pca_dim < X_tr.shape[1]:
            dr_method = DEPTH_DR_METHOD.lower()

            if dr_method == 'kernel_pca':
                # ── Kernel PCA (non-linear, rbf kernel) ────────────────────
                pca_obj = KernelPCA(n_components=depth_pca_dim, kernel='rbf',
                                    random_state=42, n_jobs=-1)
                X_tr = pca_obj.fit_transform(X_tr)
                X_v  = pca_obj.transform(X_v)
                X_te = pca_obj.transform(X_te)
                print(f"    KernelPCA on depth: {RAW_FEATURE_DIMS['depth']} -> {depth_pca_dim}")

            else:
                raise ValueError(f"Unknown DEPTH_DR_METHOD: '{DEPTH_DR_METHOD}'. ")
        
        modality_train[name] = X_tr
        modality_val[name]   = X_v
        modality_test[name]  = X_te
        feature_dims[name]   = X_tr.shape[1]
    
    # Concatenate: depth | sensor | skeleton
    X_train = np.concatenate([modality_train[n] for n in MODALITY_NAMES], axis=1)
    X_val   = np.concatenate([modality_val[n]   for n in MODALITY_NAMES], axis=1)
    X_test  = np.concatenate([modality_test[n]  for n in MODALITY_NAMES], axis=1)
    
    total = sum(feature_dims.values())
    print(f"    Feature dims after processing: " + 
          " | ".join(f"{n}={feature_dims[n]}" for n in MODALITY_NAMES) +
          f" | TOTAL={total}")
    
    return X_train, X_val, X_test, feature_dims, pca_obj


def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector (legacy, used for masks)"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality in MODALITY_NAMES:
        dim = feature_dims[modality]
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def train_and_evaluate(model, train_loader, val_loader, test_loader, num_epochs, lr):
    """Train model and return metrics"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        # Validation
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
        test_acc = accuracy_score(test_true, test_preds)
    
    return val_acc, test_acc

def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined (with per-modality normalization + depth PCA)")


Helper functions defined (with per-modality normalization + depth PCA)


## Enhanced Evaluation (returns full metrics per fold)

In [7]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Track GPU memory before training
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Feature Selection Methods
### Standard Methods (Mutual Info, RFE, LASSO)

In [8]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes, target_pct=0.5, total_features=None, feature_dims=None):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    # Calculate mutual information scores
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    # Select top k features (target percentage)
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * target_pct)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes, target_pct=0.5, total_features=None, feature_dims=None):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    # Use Random Forest as base estimator
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    # Select top k features
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * target_pct)
    selector = RFE(estimator, n_features_to_select=k, step=50)  # Remove 50 features at a time
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes, target_pct=0.5, total_features=None, feature_dims=None):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    # Train Lasso to get feature importance scores
    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='saga',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    # Rank features by coefficient magnitude across all classes
    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    # Select top-k features to match target percentage
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    target_count = int(n_feats * target_pct)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")


Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Per-Fold Experiment Runner

In [9]:
def run_all_methods_per_fold():
    """
    Run baseline + 3 standard methods across ALL folds and ALL target percentages.
    Returns master_results[method_name][target_pct][fold_idx] = {...}
    """
    print("="*80)
    print("STARTING PER-FOLD EXPERIMENTS (Standard Methods Only)")
    print(f"Methods: {ALL_METHODS}")
    print(f"Target percentages: {TARGET_FEATURE_PERCENTAGES}")
    print(f"Folds: {N_FOLDS}")
    print(f"Per-modality normalization: ENABLED")
    print(f"Depth PCA: {'ENABLED -> ' + str(DEPTH_PCA_DIM) if DEPTH_PCA_DIM else 'DISABLED'}")
    print("="*80)

    num_classes = len(np.unique(y))

    # master_results[method_name][target_pct][fold_idx] = fold_result
    master_results = {m: {p: {} for p in TARGET_FEATURE_PERCENTAGES} for m in ALL_METHODS}

    n_samples = len(y)
    dummy_X = np.zeros((n_samples, 1))
    gkf = GroupKFold(n_splits=N_FOLDS)

    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(dummy_X, y, groups=subjects)):
        print(f"\n{'#'*80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*80}")

        train_val_subjects = np.unique(subjects[train_val_idx])
        val_subject = train_val_subjects[-1]
        val_mask_in_tv   = subjects[train_val_idx] == val_subject
        train_mask_in_tv = ~val_mask_in_tv
        train_idx = train_val_idx[train_mask_in_tv]
        val_idx   = train_val_idx[val_mask_in_tv]

        print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

        X_train, X_val, X_test, fold_feature_dims, pca_obj = prepare_fold_data_per_modality(
            X_per_modality, train_idx, val_idx, test_idx,
            depth_pca_dim=DEPTH_PCA_DIM
        )
        fold_total_features = sum(fold_feature_dims.values())
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)

        for target_pct in TARGET_FEATURE_PERCENTAGES:
            print(f"\n  === Target: {target_pct*100:.0f}% ===")

            for method_name in ALL_METHODS:
                print(f"\n    --- {method_name.upper()} (target={target_pct*100:.0f}%) ---")

                fs_start_time = time.time()

                if method_name == 'baseline':
                    feature_mask = np.ones(fold_total_features, dtype=bool)
                    fs_results = {
                        'mask': feature_mask,
                        'execution_time': 0,
                        'num_selected': fold_total_features,
                        'num_total': fold_total_features,
                        'method': 'baseline'
                    }
                elif method_name == 'mutual_info':
                    fs_results = run_mutual_info_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                                    target_pct=target_pct,
                                                    total_features=fold_total_features, feature_dims=fold_feature_dims)
                    feature_mask = fs_results['mask']
                elif method_name == 'rfe':
                    fs_results = run_rfe_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                            target_pct=target_pct,
                                            total_features=fold_total_features, feature_dims=fold_feature_dims)
                    feature_mask = fs_results['mask']
                elif method_name == 'lasso':
                    fs_results = run_lasso_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                              target_pct=target_pct,
                                              total_features=fold_total_features, feature_dims=fold_feature_dims)
                    feature_mask = fs_results['mask']
                else:
                    raise ValueError(f"Unknown method: {method_name}")

                fs_time = time.time() - fs_start_time

                X_train_sel = X_train[:, feature_mask]
                X_val_sel   = X_val[:, feature_mask]
                X_test_sel  = X_test[:, feature_mask]
                opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)

                model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)

                train_loader = DataLoader(MultiModalDataset(X_train_sel, y[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
                val_loader   = DataLoader(MultiModalDataset(X_val_sel,   y[val_idx]),   batch_size=BATCH_SIZE)
                test_loader  = DataLoader(MultiModalDataset(X_test_sel,  y[test_idx]),  batch_size=BATCH_SIZE)

                if torch.cuda.is_available():
                    torch.cuda.reset_peak_memory_stats()

                eval_metrics = train_and_evaluate_full(
                    model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
                )

                if method_name != 'baseline':
                    modality_ret = fs_results.get('modality_retention',
                        calculate_modality_retention(feature_mask, fold_feature_dims))
                else:
                    modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0}
                                   for m, d in fold_feature_dims.items()}

                fold_result = {
                    'fold': fold_idx,
                    'method': method_name,
                    'target_pct': target_pct,
                    'num_train': len(train_idx),
                    'num_val': len(val_idx),
                    'num_test': len(test_idx),
                    'val_acc': eval_metrics['val_acc'],
                    'test_acc': eval_metrics['test_acc'],
                    'test_f1_macro': eval_metrics['test_f1_macro'],
                    'test_f1_weighted': eval_metrics['test_f1_weighted'],
                    'test_precision_macro': eval_metrics['test_precision_macro'],
                    'test_recall_macro': eval_metrics['test_recall_macro'],
                    'num_features_selected': int(np.sum(feature_mask)),
                    'num_features_total': fold_total_features,
                    'feature_retention_pct': float(np.sum(feature_mask) / fold_total_features * 100),
                    'modality_retention': modality_ret,
                    'feature_mask': feature_mask,
                    'feature_dims': fold_feature_dims,
                    'fs_execution_time': fs_results.get('execution_time', 0),
                    'train_time_sec': eval_metrics['train_time_sec'],
                    'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                    'model_params': eval_metrics['model_params'],
                    'model_size_mb': eval_metrics['model_size_mb'],
                    'original_dataset_size_mb': orig_dataset_size_mb,
                    'optimized_dataset_size_mb': opt_dataset_size_mb,
                    'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                    'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                    'depth_pca_dim': DEPTH_PCA_DIM,
                    'depth_pca_variance_explained': float(pca_obj.explained_variance_ratio_.sum()) if (pca_obj is not None and hasattr(pca_obj, 'explained_variance_ratio_')) else None,
                }

                master_results[method_name][target_pct][fold_idx] = fold_result

                # Save per-fold result
                pct_label = f"pct{int(target_pct*100):02d}"
                fold_dir = RESULTS_ROOT / method_name
                fold_dir.mkdir(exist_ok=True)
                fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
                fold_save_clean = {}
                for k, v in fold_save.items():
                    if isinstance(v, np.ndarray):
                        fold_save_clean[k] = v.tolist()
                    elif isinstance(v, (np.floating, np.integer)):
                        fold_save_clean[k] = float(v)
                    elif isinstance(v, dict):
                        fold_save_clean[k] = {}
                        for kk, vv in v.items():
                            if isinstance(vv, dict):
                                fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                            else:
                                fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                    else:
                        fold_save_clean[k] = v

                with open(fold_dir / f"fold_{fold_idx+1}_{pct_label}.json", 'w') as f:
                    json_lib.dump(fold_save_clean, f, indent=2, default=str)

                np.save(fold_dir / f"fold_{fold_idx+1}_{pct_label}_mask.npy", feature_mask)

                print(f"      Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                      f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                      f"Features: {np.sum(feature_mask)}/{fold_total_features} "
                      f"({np.sum(feature_mask)/fold_total_features*100:.1f}%)")

                del model, train_loader, val_loader, test_loader
                torch.cuda.empty_cache()
                gc.collect()

    print(f"\n{'='*80}")
    print("ALL EXPERIMENTS COMPLETED")
    print(f"{'='*80}")

    return master_results

print("Per-fold experiment runner defined")


Per-fold experiment runner defined


## Run All Experiments

In [10]:
master_results = run_all_methods_per_fold()

STARTING PER-FOLD EXPERIMENTS (Standard Methods Only)
Methods: ['baseline', 'mutual_info', 'rfe', 'lasso']
Target percentages: [0.1, 0.25, 0.5, 0.75, 0.9]
Folds: 5
Per-modality normalization: ENABLED
Depth PCA: ENABLED -> 512

################################################################################
# FOLD 1/5
################################################################################
  Train: 681, Val: 198, Test: 286
    KernelPCA on depth: 5508 -> 512
    Feature dims after processing: depth=512 | sensor=652 | skeleton=1879 | TOTAL=3043

  === Target: 10% ===

    --- BASELINE (target=10%) ---
      Test Acc: 97.20%, F1: 97.17%, Features: 3043/3043 (100.0%)

    --- MUTUAL_INFO (target=10%) ---
    Running Mutual Information...
      Selected: 304/3043 (10.0%), Time: 35.2s
      Test Acc: 96.15%, F1: 96.08%, Features: 304/3043 (10.0%)

    --- RFE (target=10%) ---
    Running RFE...
      Selected: 304/3043 (10.0%), Time: 8.8s
      Test Acc: 97.55%, F1: 97.56%, Features:

## Build Per-Fold Results CSV

In [11]:
# Build DataFrame with every fold x method x target_pct combination
FEATURE_DIMS = None
TOTAL_FEATURES = None

rows = []

for method_name in ALL_METHODS:
    for target_pct in TARGET_FEATURE_PERCENTAGES:
        for fold_idx in range(N_FOLDS):
            if fold_idx not in master_results[method_name][target_pct]:
                continue
            r = master_results[method_name][target_pct][fold_idx]

            if FEATURE_DIMS is None and 'feature_dims' in r:
                FEATURE_DIMS = r['feature_dims']
                TOTAL_FEATURES = sum(FEATURE_DIMS.values())
                print(f"Post-PCA feature dims: {FEATURE_DIMS}, Total: {TOTAL_FEATURES}")

            row = {
                'Method': method_name,
                'Target_Pct': target_pct,
                'Target_Pct_Label': f"{int(target_pct*100)}%",
                'Fold': fold_idx + 1,
                'Test Accuracy (%)': r['test_acc'] * 100,
                'Val Accuracy (%)': r['val_acc'] * 100,
                'F1 Macro (%)': r['test_f1_macro'] * 100,
                'F1 Weighted (%)': r['test_f1_weighted'] * 100,
                'Precision Macro (%)': r['test_precision_macro'] * 100,
                'Recall Macro (%)': r['test_recall_macro'] * 100,
                'Features Selected': r['num_features_selected'],
                'Features Total': r['num_features_total'],
                'Feature Retention (%)': r['feature_retention_pct'],
                'FS Time (s)': r['fs_execution_time'],
                'Train Time (s)': r['train_time_sec'],
                'Total Time (s)': r['total_time_sec'],
                'Model Params': r['model_params'],
                'Model Size (MB)': r['model_size_mb'],
                'Orig Dataset (MB)': r['original_dataset_size_mb'],
                'Opt Dataset (MB)': r['optimized_dataset_size_mb'],
                'Dataset Reduction (%)': r['dataset_reduction_pct'],
                'GPU Peak (MB)': r['gpu_mem_peak_mb'],
                'N Train': r['num_train'],
                'N Val': r['num_val'],
                'N Test': r['num_test'],
            }

            for mod in MODALITY_NAMES:
                if mod in r.get('modality_retention', {}):
                    row[f'{mod}_retained'] = r['modality_retention'][mod]['selected']
                    row[f'{mod}_total'] = r['modality_retention'][mod]['total']
                    row[f'{mod}_retention_%'] = r['modality_retention'][mod]['percentage']

            rows.append(row)

df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS_ROOT / "all_folds_results.csv", index=False)
print(f"Results table: {df_all.shape}")
print(f"Methods: {df_all['Method'].nunique()}, Targets: {sorted(df_all['Target_Pct'].unique())}")
df_all.head()


Post-PCA feature dims: {'depth': 512, 'sensor': 652, 'skeleton': 1879}, Total: 3043
Results table: (100, 34)
Methods: 4, Targets: [np.float64(0.1), np.float64(0.25), np.float64(0.5), np.float64(0.75), np.float64(0.9)]


,Method,Target_Pct,Target_Pct_Label,Fold,Test Accuracy (%),Val Accuracy (%),F1 Macro (%),F1 Weighted (%),Precision Macro (%),Recall Macro (%),...,N Test,depth_retained,depth_total,depth_retention_%,sensor_retained,sensor_total,sensor_retention_%,skeleton_retained,skeleton_total,skeleton_retention_%
0,baseline,0.1,10%,1,97.202797,99.494949,97.171348,97.171348,97.753533,97.202797,...,286,512,512,100.0,652,652,100.0,1879,1879,100.0
1,baseline,0.1,10%,2,92.116183,100.000000,92.261194,92.250921,93.652359,92.107438,...,241,512,512,100.0,652,652,100.0,1879,1879,100.0
2,baseline,0.1,10%,3,94.090909,100.000000,92.544370,92.544370,92.132867,94.090909,...,220,512,512,100.0,652,652,100.0,1879,1879,100.0
3,baseline,0.1,10%,4,93.181818,100.000000,92.789322,92.789322,94.405594,93.181818,...,220,512,512,100.0,652,652,100.0,1879,1879,100.0
4,baseline,0.1,10%,5,99.494949,93.775934,99.493386,99.493386,99.545455,99.494949,...,198,512,512,100.0,652,652,100.0,1879,1879,100.0


## Summary: Average per Method × Target Percentage (across all folds)

In [12]:
# Mean across folds for each (method, target_pct) combination
summary_rows = []
for method_name in ALL_METHODS:
    for target_pct in TARGET_FEATURE_PERCENTAGES:
        subset = df_all[(df_all['Method'] == method_name) & (df_all['Target_Pct'] == target_pct)]
        if len(subset) == 0:
            continue
        row = {
            'Method': method_name,
            'Target_Pct': target_pct,
            'Target_Pct_Label': f"{int(target_pct*100)}%",
            'Mean Test Acc (%)': subset['Test Accuracy (%)'].mean(),
            'Std Test Acc (%)': subset['Test Accuracy (%)'].std(),
            'Mean F1 Macro (%)': subset['F1 Macro (%)'].mean(),
            'Std F1 Macro (%)': subset['F1 Macro (%)'].std(),
            'Mean F1 Weighted (%)': subset['F1 Weighted (%)'].mean(),
            'Mean Features Selected': subset['Features Selected'].mean(),
            'Mean Feature Retention (%)': subset['Feature Retention (%)'].mean(),
            'Mean FS Time (s)': subset['FS Time (s)'].mean(),
            'Mean Train Time (s)': subset['Train Time (s)'].mean(),
            'Mean Total Time (s)': subset['Total Time (s)'].mean(),
            'Mean Model Params': subset['Model Params'].mean(),
            'Mean Model Size (MB)': subset['Model Size (MB)'].mean(),
            'Mean Dataset Reduction (%)': subset['Dataset Reduction (%)'].mean(),
            'Mean GPU Peak (MB)': subset['GPU Peak (MB)'].mean(),
            'N_Folds': len(subset),
        }
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(3)
df_summary.to_csv(RESULTS_ROOT / "summary_by_method_and_target.csv", index=False)

pct_order = [f"{int(p*100)}%" for p in TARGET_FEATURE_PERCENTAGES]

pivot_acc_mean = df_summary.pivot_table(
    values='Mean Test Acc (%)', index='Method', columns='Target_Pct_Label'
).reindex(ALL_METHODS)[pct_order]

pivot_acc_std = df_summary.pivot_table(
    values='Std Test Acc (%)', index='Method', columns='Target_Pct_Label'
).reindex(ALL_METHODS)[pct_order]

pivot_acc_combined = pivot_acc_mean.astype(str) + " ± " + pivot_acc_std.astype(str)

print("\n=== TEST ACCURACY (%) (mean ± std) ===")
print(pivot_acc_combined.to_string())


pivot_f1_mean = df_summary.pivot_table(
    values='Mean F1 Macro (%)', index='Method', columns='Target_Pct_Label'
).reindex(ALL_METHODS)[pct_order]

pivot_f1_std = df_summary.pivot_table(
    values='Std F1 Macro (%)', index='Method', columns='Target_Pct_Label'
).reindex(ALL_METHODS)[pct_order]

pivot_f1_combined = pivot_f1_mean.astype(str) + " ± " + pivot_f1_std.astype(str)

print("\n=== F1 MACRO (%) (mean ± std) ===")
print(pivot_f1_combined.to_string())



=== TEST ACCURACY (%) (mean ± std) ===
Target_Pct_Label             10%             25%             50%             75%             90%
Method                                                                                          
baseline          95.217 ± 3.052  96.848 ± 2.758   96.302 ± 2.58   95.069 ± 4.14  95.905 ± 3.946
mutual_info       93.177 ± 4.763  95.623 ± 4.636  96.037 ± 3.076  96.396 ± 3.762  97.082 ± 2.515
rfe               96.659 ± 3.193  94.483 ± 5.455   95.698 ± 2.86  95.989 ± 2.931   95.547 ± 2.96
lasso             76.363 ± 5.039  91.833 ± 5.172  93.885 ± 4.355  93.339 ± 4.382   95.366 ± 2.88

=== F1 MACRO (%) (mean ± std) ===
Target_Pct_Label             10%             25%             50%             75%             90%
Method                                                                                          
baseline          94.852 ± 3.287  96.846 ± 2.717  96.299 ± 2.521   94.623 ± 4.72  95.714 ± 4.267
mutual_info       93.046 ± 4.493   95.327 ± 5.03   9

## Visualizations

In [13]:
# ============================================================================
# PLOT 1: Heatmap — Mean Test Accuracy per Method × Target Percentage
# ============================================================================
pivot = df_summary.pivot_table(values='Mean Test Acc (%)', index='Method', columns='Target_Pct_Label')
pct_order = [f"{int(p*100)}%" for p in TARGET_FEATURE_PERCENTAGES]
pivot = pivot.reindex(ALL_METHODS)[pct_order]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Mean Test Accuracy (%)'})
ax.set_title('Mean Test Accuracy (%) per Method × Target Feature %', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Target Feature %')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_accuracy_heatmap_by_target.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_accuracy_heatmap_by_target.png")


Saved: 01_accuracy_heatmap_by_target.png


In [14]:
# ============================================================================
# PLOT 2: Line plot — Accuracy vs Target Percentage per Method
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'baseline': 'black', 'mutual_info': 'steelblue', 'rfe': 'darkorange', 'lasso': 'seagreen'}
for method in ALL_METHODS:
    mdf = df_summary[df_summary['Method'] == method].sort_values('Target_Pct')
    if len(mdf) == 0:
        continue
    x = mdf['Target_Pct'] * 100
    y_mean = mdf['Mean Test Acc (%)']
    y_std  = mdf['Std Test Acc (%)']
    ls = '--' if method == 'baseline' else '-'
    ax.plot(x, y_mean, marker='o', label=method, color=colors.get(method), linestyle=ls, linewidth=2)
    ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.15, color=colors.get(method))

ax.set_xlabel('Target Feature Retention (%)', fontsize=12)
ax.set_ylabel('Mean Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Target Feature % (mean ± std across folds)', fontsize=14)
ax.set_xticks([int(p*100) for p in TARGET_FEATURE_PERCENTAGES])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_accuracy_vs_target_pct.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_accuracy_vs_target_pct.png")


Saved: 02_accuracy_vs_target_pct.png


In [15]:
# ============================================================================
# PLOT 3: Box plot — per-fold accuracy distribution for each (method, target)
# ============================================================================
# Filter out baseline (constant across all targets)
plot_df = df_all[df_all['Method'] != 'baseline'].copy()
plot_df['Method_Target'] = plot_df['Method'] + '\n' + plot_df['Target_Pct_Label']

fig, ax = plt.subplots(figsize=(18, 7))
order = [f"{m}\n{p}" for m in ['mutual_info', 'rfe', 'lasso']
         for p in [f"{int(p*100)}%" for p in TARGET_FEATURE_PERCENTAGES]]

sns.boxplot(data=plot_df, x='Method_Target', y='Test Accuracy (%)', order=order,
            palette='Set2', ax=ax)
sns.stripplot(data=plot_df, x='Method_Target', y='Test Accuracy (%)', order=order,
              color='black', alpha=0.5, size=4, jitter=True, ax=ax)
ax.set_title('Test Accuracy Distribution per Method × Target % (across folds)', fontsize=14)
ax.set_xlabel('')
plt.xticks(rotation=60, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_accuracy_boxplot_by_target.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_accuracy_boxplot_by_target.png")


Saved: 03_accuracy_boxplot_by_target.png


In [16]:
# ============================================================================
# PLOT 4: Feature retention per method per target (verify actual vs requested)
# ============================================================================
fig, ax = plt.subplots(figsize=(10, 6))
for method in ['mutual_info', 'rfe', 'lasso']:
    mdf = df_summary[df_summary['Method'] == method].sort_values('Target_Pct')
    ax.plot(mdf['Target_Pct'] * 100, mdf['Mean Feature Retention (%)'],
            marker='s', label=method, linewidth=2)

# Ideal diagonal
x_ideal = [p*100 for p in TARGET_FEATURE_PERCENTAGES]
ax.plot(x_ideal, x_ideal, 'k--', alpha=0.4, label='Ideal (exact target)')
ax.set_xlabel('Target Feature % (requested)', fontsize=12)
ax.set_ylabel('Actual Mean Feature Retention (%)', fontsize=12)
ax.set_title('Requested vs Actual Feature Retention per Method', fontsize=14)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_retention_vs_target.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_retention_vs_target.png")


Saved: 04_retention_vs_target.png


In [17]:
# ============================================================================
# PLOT 5: Per-fold accuracy lines — one subplot per target %, all methods
# ============================================================================
fig, axes = plt.subplots(1, len(TARGET_FEATURE_PERCENTAGES), figsize=(20, 5), sharey=True)
colors = {'baseline': 'black', 'mutual_info': 'steelblue', 'rfe': 'darkorange', 'lasso': 'seagreen'}

for ax, target_pct in zip(axes, TARGET_FEATURE_PERCENTAGES):
    for method in ALL_METHODS:
        sub = df_all[(df_all['Method'] == method) & (df_all['Target_Pct'] == target_pct)].sort_values('Fold')
        if len(sub) == 0:
            continue
        lw = 2.5 if method == 'baseline' else 1.5
        ls = '--' if method == 'baseline' else '-'
        ax.plot(sub['Fold'], sub['Test Accuracy (%)'], marker='o', markersize=4,
                label=method, color=colors.get(method), linewidth=lw, linestyle=ls)
    ax.set_title(f"Target: {int(target_pct*100)}%", fontsize=11)
    ax.set_xlabel('Fold')
    ax.set_xticks(range(1, N_FOLDS + 1))
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Test Accuracy (%)')
axes[-1].legend(fontsize=8, loc='lower right')
fig.suptitle('Per-Fold Test Accuracy by Target Feature %', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_per_fold_lines_by_target.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_per_fold_lines_by_target.png")


Saved: 05_per_fold_lines_by_target.png


In [18]:
# ============================================================================
# FINAL SUMMARY: Average test accuracy per target % (averaged across all folds)
# ============================================================================
print("\n" + "="*80)
print("AVERAGE TEST ACCURACY (%) PER TARGET % (mean ± std across 8 folds)")
print("="*80)

pct_labels = [f"{int(p*100)}%" for p in TARGET_FEATURE_PERCENTAGES]
for method in ALL_METHODS:
    print(f"\n  {method.upper()}")
    for target_pct in TARGET_FEATURE_PERCENTAGES:
        subset = df_all[(df_all['Method'] == method) & (df_all['Target_Pct'] == target_pct)]
        if len(subset) == 0:
            continue
        mean_acc = subset['Test Accuracy (%)'].mean()
        std_acc  = subset['Test Accuracy (%)'].std()
        mean_f1  = subset['F1 Macro (%)'].mean()
        mean_ret = subset['Feature Retention (%)'].mean()
        print(f"    Target {int(target_pct*100):3d}%: "
              f"Acc={mean_acc:.2f}% ±{std_acc:.2f}  |  "
              f"F1={mean_f1:.2f}%  |  "
              f"Retention={mean_ret:.2f}%")

print("\n" + "="*80)
print("SAVED FILES:")
print(f"  {RESULTS_ROOT / 'all_folds_results.csv'}")
print(f"  {RESULTS_ROOT / 'summary_by_method_and_target.csv'}")
print("="*80)



AVERAGE TEST ACCURACY (%) PER TARGET % (mean ± std across 8 folds)

  BASELINE
    Target  10%: Acc=95.22% ±3.05  |  F1=94.85%  |  Retention=100.00%
    Target  25%: Acc=96.85% ±2.76  |  F1=96.85%  |  Retention=100.00%
    Target  50%: Acc=96.30% ±2.58  |  F1=96.30%  |  Retention=100.00%
    Target  75%: Acc=95.07% ±4.14  |  F1=94.62%  |  Retention=100.00%
    Target  90%: Acc=95.90% ±3.95  |  F1=95.71%  |  Retention=100.00%

  MUTUAL_INFO
    Target  10%: Acc=93.18% ±4.76  |  F1=93.05%  |  Retention=9.99%
    Target  25%: Acc=95.62% ±4.64  |  F1=95.33%  |  Retention=24.98%
    Target  50%: Acc=96.04% ±3.08  |  F1=95.71%  |  Retention=49.98%
    Target  75%: Acc=96.40% ±3.76  |  F1=96.34%  |  Retention=74.99%
    Target  90%: Acc=97.08% ±2.52  |  F1=97.12%  |  Retention=89.98%

  RFE
    Target  10%: Acc=96.66% ±3.19  |  F1=96.71%  |  Retention=9.99%
    Target  25%: Acc=94.48% ±5.45  |  F1=94.27%  |  Retention=24.98%
    Target  50%: Acc=95.70% ±2.86  |  F1=95.64%  |  Retention=49.98